# $\mathbb{Z}_2^F \times \mathbb{Z}_2^T$ states - debugging

Created: 28-07-2026

Objectives:
* Having repeated issues with these states. Be thorough and check expected properties.

# Imports

In [1]:
import numpy as np
import pandas as pd

In [2]:
import jax
jax.config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from operator import mul
from itertools import product

In [6]:
import quimb.tensor as qtn
import quimb as qu

In [7]:
from quspin.operators import hamiltonian
from quspin.operators import quantum_operator
from quspin.basis import spin_basis_1d, spinless_fermion_basis_1d, tensor_basis

In [8]:
from time import time

In [9]:
from humanize import naturalsize

# Load state

In [17]:
L = 4

In [18]:
spin_basis = spin_basis_1d(L)
fermion_basis = spinless_fermion_basis_1d(L)
basis = tensor_basis(spin_basis, fermion_basis)

In [19]:
TRIV_COCYCLE_DIR = r"../../data/z2_f_x_z2_t_triv_to_nontriv_n1_4_site_ed"

In [20]:
triv_energies = [
    np.load(rf'{TRIV_COCYCLE_DIR}/{i}.npz')['energy']
    for i in range(0, 101, 5)
]

In [21]:
hamiltonian_psi = np.load(rf'{TRIV_COCYCLE_DIR}/100.npz')['psi'][:, 0]

In [22]:
print(basis)

reference states: 
array index   /   Fock state   /   integer repr. 
	  0.  |1 1 1 1>  15 |1 1 1 1>  15 
	  1.  |1 1 1 1>  15 |1 1 1 0>  14 
	  2.  |1 1 1 1>  15 |1 1 0 1>  13 
	  3.  |1 1 1 1>  15 |1 1 0 0>  12 
	  4.  |1 1 1 1>  15 |1 0 1 1>  11 
	  5.  |1 1 1 1>  15 |1 0 1 0>  10 
	  6.  |1 1 1 1>  15 |1 0 0 1>   9 
	  7.  |1 1 1 1>  15 |1 0 0 0>   8 
	  8.  |1 1 1 1>  15 |0 1 1 1>   7 
	  9.  |1 1 1 1>  15 |0 1 1 0>   6 
	 10.  |1 1 1 1>  15 |0 1 0 1>   5 
	 11.  |1 1 1 1>  15 |0 1 0 0>   4 
	 12.  |1 1 1 1>  15 |0 0 1 1>   3 
	 13.  |1 1 1 1>  15 |0 0 1 0>   2 
	 14.  |1 1 1 1>  15 |0 0 0 1>   1 
	 15.  |1 1 1 1>  15 |0 0 0 0>   0 
	 16.  |1 1 1 0>  14 |1 1 1 1>  15 
	 17.  |1 1 1 0>  14 |1 1 1 0>  14 
	 18.  |1 1 1 0>  14 |1 1 0 1>  13 
	 19.  |1 1 1 0>  14 |1 1 0 0>  12 
	 20.  |1 1 1 0>  14 |1 0 1 1>  11 
	 21.  |1 1 1 0>  14 |1 0 1 0>  10 
	 22.  |1 1 1 0>  14 |1 0 0 1>   9 
	 23.  |1 1 1 0>  14 |1 0 0 0>   8 
	 24.  |1 1 1 0>  14 |0 1 1 1>   7 
                :
	231.  |0 0 0

In [23]:
local_stab = quantum_operator(
    { "_": [
        ["z|--", [[1, 2, 1, 2]]],
        ["z|-+", [[1, 2, 1, 2]]],
        ["z|+-", [[1, 2, 1, 2]]],
        ["z|++", [[1, 2, 1, 2]]],
    ]},
    basis=basis,
    check_herm=False
)

/tmp/ipykernel_20611/906443670.py:1: UserWarning: Test for symmetries not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_symm=False in hamiltonian
  local_stab = quantum_operator(
/tmp/ipykernel_20611/906443670.py:1: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  local_stab = quantum_operator(


In [24]:
np.abs(local_stab.expt_value(hamiltonian_psi))

np.float64(1.3877787807814457e-17)

## Minimal example - for chatgpt

In [35]:
from quspin.operators import hamiltonian
from quspin.operators import quantum_operator
from quspin.basis import spin_basis_1d, spinless_fermion_basis_1d, tensor_basis

group_quads = list(product([0,1], repeat=4))

spin_ops_dict = {
    (0,0): [("I", 1), ("y", 1)],
    (1,1): [("I", 1), ("y", -1)],
    (0,1): [("z", 1), ("x", 1j)],
    (1,0): [("z", 1), ("x", -1j)]
}

def get_fermionic_op_string(group_quad):
    g_left, g_in, g_out, g_right = group_quad

    out_string = ''
    out_indices = list()

    if (g_left + g_out) % 2:
        out_string += '+'
        out_indices.append(0)
    if (g_out + g_right) % 2:
        out_string += '+'
        out_indices.append(1)
    if (g_in + g_right) % 2:
        out_string += '-'
        out_indices.append(1)
    if (g_left + g_in) % 2:
        out_string += '-'
        out_indices.append(0)

    return (out_string, out_indices)

def get_group_quad_terms(group_quad, L):
    g_left, g_in, g_out, g_right = group_quad

    left_op = spin_ops_dict[(g_left, g_left)]
    mid_op = spin_ops_dict[(g_out, g_in)]
    right_op = spin_ops_dict[(g_right, g_right)]

    op_triples = product(left_op, mid_op, right_op)

    terms = list()

    for left_pair, mid_pair, right_pair in op_triples:
        left_string, left_strength = left_pair
        mid_string, mid_strength = mid_pair
        right_string, right_strength = right_pair

        spin_string = f"{left_string}{mid_string}{right_string}"
        strength = -(1/16)*left_strength*mid_strength*right_strength

        ferm_op_string, ferm_indices = get_fermionic_op_string(group_quad)
        op_string = f"{spin_string}|{ferm_op_string}"

        base_index = [0, 1, 2, *ferm_indices]
        all_indices = [
            [(x+i)%L for x in base_index]
            for i in range(L)
        ]

        current_term = [
            op_string, [[strength, *indices] for indices in all_indices]
        ]

        terms.append(current_term)

    return terms

spin_basis = spin_basis_1d(L)
fermion_basis = spinless_fermion_basis_1d(L)
basis = tensor_basis(spin_basis, fermion_basis)

terms = [
    l for group_quad in group_quads
    for l in get_group_quad_terms(group_quad, L)
]

h = hamiltonian(
    terms,
    [],
    basis=basis,
    dtype=np.complex128,
    check_symm=False,
    check_herm=False
)

e, psi = h.eigsh(k=1, which='SA')

local_stab = quantum_operator(
    { "_": [
        ["z|--", [[1, 2, 1, 2]]],
        ["z|-+", [[1, 2, 1, 2]]],
        ["z|+-", [[1, 2, 1, 2]]],
        ["z|++", [[1, 2, 1, 2]]],
    ]},
    basis=basis,
    check_herm=False
)

out=np.abs(local_stab.expt_value(psi))

/tmp/ipykernel_20611/2610076766.py:80: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  h = hamiltonian(
/tmp/ipykernel_20611/2610076766.py:91: UserWarning: Test for symmetries not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_symm=False in hamiltonian
  local_stab = quantum_operator(
/tmp/ipykernel_20611/2610076766.py:91: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  local_stab = quantum_operator(


In [39]:
comm = h.static @ (local_stab.tocsr()) - local_stab.tocsr() @ (h.static)
print(np.abs(comm).max())

0.875


So it's not a symmetry... why?

Construct a local projector:

In [50]:
def get_group_quad_terms_single_site(group_quad, L, site):
    g_left, g_in, g_out, g_right = group_quad

    left_op = spin_ops_dict[(g_left, g_left)]
    mid_op = spin_ops_dict[(g_out, g_in)]
    right_op = spin_ops_dict[(g_right, g_right)]

    op_triples = product(left_op, mid_op, right_op)

    terms = list()

    for left_pair, mid_pair, right_pair in op_triples:
        left_string, left_strength = left_pair
        mid_string, mid_strength = mid_pair
        right_string, right_strength = right_pair

        spin_string = f"{left_string}{mid_string}{right_string}"
        strength = (1/16)*left_strength*mid_strength*right_strength

        ferm_op_string, ferm_indices = get_fermionic_op_string(group_quad)
        op_string = f"{spin_string}|{ferm_op_string}"

        base_index = [0, 1, 2, *ferm_indices]
        translated_index = [
            (x+site-1)%L for x in base_index
        ]

        current_term = [
            op_string, [[strength, *translated_index]]
        ]

        terms.append(current_term)

    return terms

In [51]:
single_site_terms = [
    l for group_quad in group_quads
    for l in get_group_quad_terms_single_site(group_quad, L, 1)
]

In [52]:
single_site_projectors = list()

for i in range(4):
    single_site_terms = [
        l for group_quad in group_quads
        for l in get_group_quad_terms_single_site(group_quad, L, i)
    ]

    op = quantum_operator(
        {"_": single_site_terms},
        basis=basis,
        check_herm=False
    )

    single_site_projectors.append(op)

/tmp/ipykernel_20611/2931135295.py:9: UserWarning: Test for symmetries not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_symm=False in hamiltonian
  op = quantum_operator(
/tmp/ipykernel_20611/2931135295.py:9: UserWarning: Test for particle conservation not implemented for <class 'quspin.basis.tensor.tensor_basis'>, to turn off this warning set check_pcon=False in hamiltonian
  op = quantum_operator(


In [53]:
[
    op.expt_value(psi)
    for op in single_site_projectors
]

[array([1.-1.73472348e-18j]),
 array([1.-8.67361738e-19j]),
 array([1.+2.08166817e-17j]),
 array([1.+3.46944695e-18j])]

In [56]:
op.matmat(op)

TypeError: must be real number, not quantum_operator

In [ ]:
comm = h.static @ (local_stab.tocsr()) - local_stab.tocsr() @ (h.static)
print(np.abs(comm).max())

In [57]:
check = op.tocsr() @ op.tocsr() - op.tocsr()
print(np.abs(check).max())

0.1875


Not a projector...!

In [58]:
for op in single_site_projectors:
    check = op.tocsr() @ op.tocsr() - op.tocsr()
    print(np.abs(check).max())

0.1875
0.1875
0.1875
0.1875
